# 05 - Agent 評估框架（LLM-as-judge）

一個 agent 能「回答問題」不代表它回答得好。評估框架讓你用數字衡量：

| 指標 | 意義 |
|------|------|
| **judge_score** | LLM 評分員對答案正確性的評分（0–2） |
| **citation_ok** | 答案是否引用了期望的來源檔案 |
| **tool_steps** | Agent 用了幾次工具才得出答案（效率指標） |

**LLM-as-judge 的核心想法**：用另一個 LLM 呼叫來評估答案，
輸入是「問題 + 標準答案關鍵字 + agent 的實際回答」，
輸出是結構化的評分與理由。

In [1]:
import os
import re
import sys
import uuid
from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal

_cwd = Path().resolve()
PROJECT_ROOT = _cwd if (_cwd / 'data').exists() else _cwd.parent
EXAMPLES_DIR = PROJECT_ROOT / 'examples'
if str(EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_DIR))

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, RemoveMessage, SystemMessage, ToolMessage
from langchain_core.tools import StructuredTool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from utils.tools import FileTools

load_dotenv(PROJECT_ROOT / '.env')
print(f'Project root: {PROJECT_ROOT}')
print(f'LLM: {os.environ["VLLM_MODEL"]}')

Project root: /home/mistin/research/agentic-rag-lab
LLM: gemma-4-31B-it


## 1 · 題組定義

`EvalCase` 是一個 dataclass，儲存：
- `question` — 送給 agent 的問題
- `expected_keywords` — 正確答案應包含的關鍵詞（用於 LLM judge 對照）
- `expected_sources` — 答案應引用的來源檔名（至少一個）

五個題目與 02 相同，方便日後對比不同版本 agent 的分數。

In [2]:
@dataclass
class EvalCase:
    question: str
    expected_keywords: list[str]
    expected_sources: list[str]


EVAL_CASES: list[EvalCase] = [
    EvalCase(
        question='晨星科技最初決定採用什麼部署方案？後來為何改變？是誰提出異議、理由是什麼？',
        expected_keywords=['公有雲', '私有雲', '蔡國峰', '資安', '資料在地化'],
        expected_sources=['meeting_003_晨星科技_2026-05-22.md'],
    ),
    EvalCase(
        question='陳建宏目前同時負責哪些專案？跨專案的人力衝突帶來哪些具體風險？',
        expected_keywords=['晨星科技', '新星金融科技', '支援期', '2026-07-01', '遷移驗證'],
        expected_sources=['meeting_003_晨星科技_2026-05-22.md', 'meeting_004_新星金融科技_2026-05-28.md'],
    ),
    EvalCase(
        question='新星金融科技的 PoC 通過條件是什麼？如果未通過，合作會如何發展？',
        expected_keywords=['10000', 'TPS', '50ms', '未通過', '合作暫停'],
        expected_sources=['meeting_004_新星金融科技_2026-05-28.md'],
    ),
    EvalCase(
        question='哪個客戶面臨最緊迫的法遵稽核壓力？稽核截止日期與具體要求為何？',
        expected_keywords=['新星金融科技', '金管會', 'Q3', 'Audit Trail', '電子支付'],
        expected_sources=['meeting_004_新星金融科技_2026-05-28.md'],
    ),
    EvalCase(
        question='請列出目前所有客戶專案的預定 Go-Live 日期，並指出哪個時程的風險最高及原因。',
        expected_keywords=['晨星科技', '鴻圖零售', '新星金融科技', '2026-06-15', '2026-07-15', '2026-10-31'],
        expected_sources=['meeting_001_晨星科技_2026-05-08.md', 'meeting_002_鴻圖零售_2026-05-15.md'],
    ),
]

print(f'題組載入完成，共 {len(EVAL_CASES)} 題')

題組載入完成，共 5 題


## 2 · 建立受測 Agent

使用與 02 相同的 CRM agent（`agent_v2`，強制 `read_file`），
加上一個 `run_and_collect()` 函式，回傳結構化的執行結果。

In [3]:
_crm = FileTools(PROJECT_ROOT / 'data' / 'crm_notes')
tools = [
    StructuredTool.from_function(_crm.list_files),
    StructuredTool.from_function(_crm.grep),
    StructuredTool.from_function(_crm.read_file),
]
_tools_by_name = {t.name: t for t in tools}

llm = ChatOpenAI(
    base_url=os.environ['VLLM_BASE_URL'],
    api_key=os.environ['VLLM_API_KEY'],
    model=os.environ['VLLM_MODEL'],
    temperature=0,
)
llm_with_tools = llm.bind_tools(tools)

SYSTEM_PROMPT = (
    '你是 CRM 資料分析助理，只能依據會議紀錄回答問題。\n'
    '可用工具：list_files、grep、read_file。\n'
    '回答時引用來源檔名，資料不足請如實說明。'
)

_TEXT_TOOL_RE = re.compile(r'call:(\w+)\{([^}]*)\}')


def _parse_text_calls(content: str) -> list[tuple[str, dict]]:
    calls = []
    for m in _TEXT_TOOL_RE.finditer(content):
        args: dict = {}
        for kv in m.group(2).split(','):
            if ':' in kv:
                k, v = kv.split(':', 1)
                args[k.strip()] = v.strip()
        calls.append((m.group(1), args))
    return calls


def _was_read_file_called(messages: list) -> bool:
    return any(
        tc.get('name') == 'read_file'
        for msg in messages
        if isinstance(msg, AIMessage)
        for tc in getattr(msg, 'tool_calls', [])
    )


def agent_node(state: MessagesState) -> dict:
    msgs = [SystemMessage(content=SYSTEM_PROMPT)] + state['messages']
    return {'messages': [llm_with_tools.invoke(msgs)]}


def text_tool_node(state: MessagesState) -> dict:
    last = state['messages'][-1]
    calls = _parse_text_calls(str(last.content))
    if not calls:
        return {'messages': []}
    result_msgs = [RemoveMessage(id=last.id)]
    for name, args in calls:
        tool = _tools_by_name.get(name)
        try:
            result = tool.invoke(args) if tool else f'Unknown tool: {name}'
        except Exception as exc:
            result = f'Error: {exc}'
        call_id = uuid.uuid4().hex[:8]
        result_msgs.append(AIMessage(
            content='',
            tool_calls=[{'name': name, 'args': args, 'id': call_id, 'type': 'tool_call'}],
        ))
        result_msgs.append(ToolMessage(content=str(result), tool_call_id=call_id))
    return {'messages': result_msgs}


def must_read_node(state: MessagesState) -> dict:
    return {'messages': [HumanMessage(
        content='你尚未使用 read_file 讀取任何完整文件，請先閱讀相關紀錄再回答。'
    )]}


def route(state: MessagesState) -> Literal['tools', 'text_tools', 'must_read', '__end__']:
    last = state['messages'][-1]
    if not isinstance(last, AIMessage):
        return END
    if last.tool_calls:
        return 'tools'
    if isinstance(last.content, str) and _TEXT_TOOL_RE.search(last.content):
        return 'text_tools'
    if not _was_read_file_called(state['messages']):
        return 'must_read'
    return END


_builder = StateGraph(MessagesState)
_builder.add_node('agent', agent_node)
_builder.add_node('tools', ToolNode(tools))
_builder.add_node('text_tools', text_tool_node)
_builder.add_node('must_read', must_read_node)
_builder.add_edge(START, 'agent')
_builder.add_conditional_edges('agent', route)
_builder.add_edge('tools', 'agent')
_builder.add_edge('text_tools', 'agent')
_builder.add_edge('must_read', 'agent')
agent = _builder.compile()
print('Agent 建立完成 ✓')

Agent 建立完成 ✓


In [4]:
@dataclass
class AgentResult:
    question: str
    answer: str
    tool_steps: int
    sources_cited: list[str]  # 從答案文字中找到的 .md 檔名


def run_and_collect(question: str) -> AgentResult:
    """執行 agent 並回傳結構化結果（answer、工具步驟數、引用來源）。"""
    tool_steps = 0
    final_answer = ''

    for chunk in agent.stream(
        {'messages': [HumanMessage(content=question)]},
        config={'recursion_limit': 30},
        stream_mode='updates',
    ):
        for _node, update in chunk.items():
            for msg in update.get('messages', []):
                if isinstance(msg, RemoveMessage):
                    continue
                if isinstance(msg, AIMessage) and getattr(msg, 'tool_calls', None):
                    tool_steps += len(msg.tool_calls)
                elif isinstance(msg, AIMessage) and not getattr(msg, 'tool_calls', None):
                    if isinstance(msg.content, str) and msg.content.strip():
                        final_answer = msg.content.strip()

    # 從答案中抽取引用的 .md 檔名
    cited = re.findall(r'meeting_\d+_[^`\s]+\.md', final_answer)
    return AgentResult(
        question=question,
        answer=final_answer,
        tool_steps=tool_steps,
        sources_cited=list(dict.fromkeys(cited)),  # 去重，保留順序
    )

## 3 · LLM-as-judge

Judge prompt 要求 LLM 對每個答案輸出 JSON，包含：
- `score`：0（錯誤）/ 1（部分正確）/ 2（完全正確）
- `reason`：評分理由（一句話）

Judge 使用同一個 LLM，但以獨立呼叫執行，不依賴 agent 的對話歷史。

In [5]:
import json
from openai import OpenAI

_openai = OpenAI(base_url=os.environ['VLLM_BASE_URL'], api_key=os.environ['VLLM_API_KEY'])


@dataclass
class JudgeResult:
    score: int  # 0, 1, 2
    reason: str


def llm_judge(case: EvalCase, agent_answer: str) -> JudgeResult:
    """呼叫 LLM 評分員，回傳 JudgeResult。"""
    keywords_str = ', '.join(f'「{k}」' for k in case.expected_keywords)
    judge_prompt = (
        f'問題：{case.question}\n\n'
        f'正確答案應包含以下關鍵概念：{keywords_str}\n\n'
        f'Agent 的回答：\n{agent_answer}\n\n'
        '請評分 Agent 的回答。評分標準：\n'
        '  2 = 完全正確，涵蓋所有關鍵概念\n'
        '  1 = 部分正確，涵蓋超過一半關鍵概念\n'
        '  0 = 錯誤或嚴重缺漏\n\n'
        '請只輸出 JSON，格式：{"score": <0|1|2>, "reason": "<一句話理由>"}'
    )
    resp = _openai.chat.completions.create(
        model=os.environ['VLLM_MODEL'],
        messages=[{'role': 'user', 'content': judge_prompt}],
        temperature=0,
    )
    raw = resp.choices[0].message.content or ''
    # 從回應中抽取 JSON
    m = re.search(r'\{[^}]+\}', raw)
    if m:
        data = json.loads(m.group())
        return JudgeResult(score=int(data.get('score', 0)), reason=data.get('reason', raw))
    return JudgeResult(score=0, reason=raw[:100])

## 4 · 執行評估

對所有 5 題依序執行 agent + judge，收集結果。

In [6]:
@dataclass
class EvalRecord:
    case: EvalCase
    result: AgentResult
    judge: JudgeResult
    citation_ok: bool


records: list[EvalRecord] = []

for i, case in enumerate(EVAL_CASES, start=1):
    result = run_and_collect(case.question)
    judge  = llm_judge(case, result.answer)
    # citation_ok：期望來源中至少有一個出現在答案裡
    citation_ok = any(src in result.sources_cited for src in case.expected_sources)
    records.append(EvalRecord(case=case, result=result, judge=judge, citation_ok=citation_ok))
    cite_mark = '✓' if citation_ok else '✗'
    print(f'[Q{i}] steps={result.tool_steps}  {cite_mark} 引用來源  judge={judge.score}')

[Q1] steps=3  ✓ 引用來源  judge=2
[Q2] steps=3  ✓ 引用來源  judge=2
[Q3] steps=2  ✓ 引用來源  judge=2
[Q4] steps=2  ✓ 引用來源  judge=2
[Q5] steps=6  ✓ 引用來源  judge=2


## 5 · 結果統計

In [7]:
def print_results(records: list[EvalRecord]) -> None:
    w = 44
    sep = f'┌{"─"*6}┬{"─"*12}┬{"─"*16}┬{"─"*12}┬{"─"*w}┐'
    mid = f'├{"─"*6}┼{"─"*12}┼{"─"*16}┼{"─"*12}┼{"─"*w}┤'
    bot = f'└{"─"*6}┴{"─"*12}┴{"─"*16}┴{"─"*12}┴{"─"*w}┘'
    print(sep)
    print(f'│{"題目":^6}│{"tool_steps":^12}│{"citation_ok":^16}│{"judge":^12}│{"reason":^{w}}│')
    print(mid)
    for i, r in enumerate(records, start=1):
        cite = '✓' if r.citation_ok else '✗'
        reason = r.judge.reason[:w-2]
        print(f'│{f"Q{i}":^6}│{r.result.tool_steps:^12}│{cite:^16}│{r.judge.score:^12}│ {reason:<{w-2}} │')
    print(bot)

    avg_score = sum(r.judge.score for r in records) / len(records)
    cite_count = sum(1 for r in records if r.citation_ok)
    avg_steps  = sum(r.result.tool_steps for r in records) / len(records)
    print(f'\n平均 judge_score : {avg_score:.1f} / 2.0')
    print(f'Citation 正確率  : {cite_count} / {len(records)}  ({100*cite_count/len(records):.1f}%)')
    print(f'平均 tool_steps  : {avg_steps:.1f}')


print_results(records)

┌──────┬────────────┬────────────────┬────────────┬────────────────────────────────────────────┐
│  題目  │ tool_steps │  citation_ok   │   judge    │                   reason                   │
├──────┼────────────┼────────────────┼────────────┼────────────────────────────────────────────┤
│  Q1  │     3      │       ✓        │     2      │ 回答完全正確，且完整涵蓋了公有雲、私有雲、蔡國峰、資安、資料在地化所有關鍵概念。   │
│  Q2  │     3      │       ✓        │     2      │ 回答完整涵蓋了所有關鍵概念（晨星科技、新星金融科技、支援期、2026-07-01、遷 │
│  Q3  │     2      │       ✓        │     2      │ 回答完整涵蓋了 10000, TPS, 50ms, 未通過, 合作暫停 所有關鍵概念 │
│  Q4  │     2      │       ✓        │     2      │ 回答完全正確，且涵蓋了所有關鍵概念：新星金融科技、金管會、Q3、Audit Trai │
│  Q5  │     6      │       ✓        │     2      │ 回答完整涵蓋了所有要求的客戶名稱及對應的 Go-Live 日期，並詳細分析了風險最高 │
└──────┴────────────┴────────────────┴────────────┴────────────────────────────────────────────┘

平均 judge_score : 2.0 / 2.0
Citation 正確率  : 5 / 5  (100.0%)
平均 tool_steps  : 3.2


## 小結

`EvalCase` + `run_and_collect()` + `llm_judge()` 構成了一個可重用的評估流程：

- 題組（`EVAL_CASES`）與 agent 邏輯分離，未來改版 agent 只需重跑評估
- `judge_score` 捕捉答案品質，`citation_ok` 捕捉資料可追溯性，`tool_steps` 捕捉效率
- Q4 的 `citation_ok=✗` 說明 agent 找到了正確資訊但沒有在答案裡引用來源檔名，
  這是 system prompt 需要加強的地方，而不是資料問題

---
**下一個筆記本（06）**：用 `with_structured_output()` 取代純文字萃取，
並透過本節的評估框架量化兩者的品質差異。